In [ ]:
import numpy as np
import pandas as pd
import os

def random_split(interaction_df: pd.DataFrame, test_ratio: float, seed: int = 42) -> tuple[pd.DataFrame, pd.DataFrame]:
    rng = np.random.RandomState(seed)

    def stochastic_round(x: float) -> int:
        lo = int(np.floor(x))
        return lo + (1 if rng.rand() < (x - lo) else 0)

    train_parts, test_parts = [], []
    for u, g in interaction_df.groupby("user_id"):
        idx = g.index.to_numpy()
        n = len(idx)

        rng.shuffle(idx)
        n_test = stochastic_round(test_ratio * n)
        test_idx = idx[:n_test]
        train_idx = idx[n_test:]

        if len(train_idx) > 0:
            train_parts.append(interaction_df.loc[train_idx])
        if len(test_idx) > 0:
            test_parts.append(interaction_df.loc[test_idx])

    train_df = pd.concat(train_parts, ignore_index=True)
    test_df = pd.concat(test_parts, ignore_index=True)
    return train_df, test_df


raw_data_path = './SHNS_data_raw/ml1m_raw'
output_data_path = './SHNS_data'


raw_df = pd.read_csv(
    raw_data_path + '/ratings.dat',
    sep="::",
    engine="python",
    header=None,
    names=["user_id", "item_id", "rating", "timestamp"],
    dtype={"user_id": "int32", "movie_id": "int32", "rating": "int8", "timestamp": "int64"},
)
raw_df = raw_df[['user_id', 'item_id']]
raw_train_valid_df, raw_test_df = random_split(raw_df, test_ratio=0.1, seed=42)
raw_train_df, raw_valid_df = random_split(raw_train_valid_df, test_ratio=0.1, seed=42)

all_df = pd.concat([raw_train_df, raw_valid_df, raw_test_df], ignore_index=True)
users = sorted(all_df['user_id'].unique())
items = sorted(all_df['item_id'].unique())
user2id = {u:i for i, u in enumerate(users)}
item2id = {a:i for i, a in enumerate(items)}

train_df = raw_train_df.copy()
train_df['user_id'] = train_df['user_id'].map(user2id)
train_df['item_id'] = train_df['item_id'].map(item2id)

valid_df = raw_valid_df.copy()
valid_df['user_id'] = valid_df['user_id'].map(user2id)
valid_df['item_id'] = valid_df['item_id'].map(item2id)

test_df = raw_test_df.copy()
test_df['user_id'] = test_df['user_id'].map(user2id)
test_df['item_id'] = test_df['item_id'].map(item2id)

cur_output_data_path = output_data_path + '/' + 'ml1m'
os.makedirs(cur_output_data_path, exist_ok=True)
if os.path.exists(cur_output_data_path + "/train.txt"):
    print(f"{cur_output_data_path + '/train.txt'} already exists. Checking sanity...")
    existing_train_df = pd.read_csv(cur_output_data_path + "/train.txt", sep="\t", header=None, names=["user_id", "item_id"])
    assert train_df.equals(existing_train_df), "Existing train.txt does not match the newly created train_df!"
else:
    train_df[["user_id", "item_id"]].to_csv(cur_output_data_path + "/train.txt", sep="\t", header=False, index=False)
if os.path.exists(cur_output_data_path + "/valid.txt"):
    print(f"{cur_output_data_path + '/valid.txt'} already exists. Checking sanity...")
    existing_valid_df = pd.read_csv(cur_output_data_path + "/valid.txt", sep="\t", header=None, names=["user_id", "item_id"])
    assert valid_df.equals(existing_valid_df), "Existing valid.txt does not match the newly created valid_df!"
else:
    valid_df[["user_id", "item_id"]].to_csv(cur_output_data_path + "/valid.txt", sep="\t", header=False, index=False)
if os.path.exists(cur_output_data_path + "/test.txt"):
    print(f"{cur_output_data_path + '/test.txt'} already exists. Checking sanity...")
    existing_test_df = pd.read_csv(cur_output_data_path + "/test.txt", sep="\t", header=None, names=["user_id", "item_id"])
    assert test_df.equals(existing_test_df), "Existing test.txt does not match the newly created test_df!"
else:
    test_df[["user_id", "item_id"]].to_csv(cur_output_data_path + "/test.txt",  sep="\t", header=False, index=False)
